# Superstore Sales Analysis

## Notebook 1: Data Preparation and Validation

## Objective

This notebook loads and validates the Orders, Returns and People tables of the superstore sales dataset.
The goal is to prepare a reusable analytical datasets for sales, profitability, customer,returns and time-series analysis.

## Dataset grain

It should be noted that the Orders table contains one row per product line within an order.
Therefore, the number of rows is not equal to the number of unique orders.

In [1]:
# PROJECT SETUP
# ============================================================
# Import Path for creating operating-system-independent file paths.

from pathlib import Path

# Importing pandas for loading and analysing tabular data.
import numpy as np

# Import NumPy for numerical checks such as np.isclose().
import pandas as pd


# To find the directory from which the notebook is running.

CURRENT_DIR = Path.cwd()


# If the notebook is running from the notebooks folder, move one level upward to locate the main project folder.

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Define input and output folder locations.

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

TABLES_DIR = (
    PROJECT_ROOT / "results" / "tables"
)

FIGURES_DIR = (
    PROJECT_ROOT / "results" / "figures"
)


# Create the output folders.


for folder in [
    PROCESSED_DATA_DIR,
    TABLES_DIR,
    FIGURES_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


In [2]:
# Checking whether all required project folders exist.

folder_check = pd.Series({
    "raw_data": RAW_DATA_DIR.exists(),
    "processed_data": PROCESSED_DATA_DIR.exists(),
    "tables": TABLES_DIR.exists(),
    "figures": FIGURES_DIR.exists()
})

folder_check

raw_data          True
processed_data    True
tables            True
figures           True
dtype: bool

In [3]:
# List only the files stored inside data/raw.
# This is to prevents filename assumptions and spelling mistakes.

raw_filenames = sorted(
    file.name
    for file in RAW_DATA_DIR.iterdir()
    if file.is_file()
)

raw_filenames

['orders.csv',
 'people.csv',
 'returns.csv',
 'sample_superstore.xlsx',
 '~$sample_superstore.xlsx']

In [4]:
# Defined the location of each raw source file.

ORDERS_FILE = (
    RAW_DATA_DIR / "sample_superstore.xlsx"
)

RETURNS_FILE = (
    RAW_DATA_DIR / "returns.csv"
)

PEOPLE_FILE = (
    RAW_DATA_DIR / "people.csv"
)

In [5]:
# Verified whether each source file exists or not

source_file_check = pd.Series({
    "orders_file": ORDERS_FILE.exists(),
    "returns_file": RETURNS_FILE.exists(),
    "people_file": PEOPLE_FILE.exists()
})

source_file_check

orders_file     True
returns_file    True
people_file     True
dtype: bool

In [6]:
#  The orders worksheet from the Excel workbook is loaded.

orders = pd.read_excel(
    ORDERS_FILE,
    sheet_name="Orders"
)


# The returns.csv file is loaded.

returns_raw = pd.read_csv(
    RETURNS_FILE
)


# The regional-manager.csv file mapping is loaded.

people = pd.read_csv(
    PEOPLE_FILE
)

In [7]:
# Inspecting the size of each source file in tabular format.

source_inventory = pd.DataFrame({
    "table": [
        "Orders",
        "Returns_raw",
        "People"
    ],
    "rows": [
        orders.shape[0],
        returns_raw.shape[0],
        people.shape[0]
    ],
    "columns": [
        orders.shape[1],
        returns_raw.shape[1],
        people.shape[1]
    ]
})

source_inventory

,table,rows,columns
0,Orders,10194,21
1,Returns_raw,296,7
2,People,4,2


In [8]:
# Inspect the Orders table and data types.
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  str           
 2   Order Date      10194 non-null  datetime64[us]
 3   Ship Date       10194 non-null  datetime64[us]
 4   Ship Mode       10194 non-null  str           
 5   Customer ID     10194 non-null  str           
 6   Customer Name   10194 non-null  str           
 7   Segment         10194 non-null  str           
 8   Country/Region  10194 non-null  str           
 9   City            10194 non-null  str           
 10  State/Province  10194 non-null  str           
 11  Postal Code     10194 non-null  object        
 12  Region          10194 non-null  str           
 13  Product ID      10194 non-null  str           
 14  Category        10194 non-null  str           
 15  Sub-Category 

In [9]:
# Display the exact column names in the secondary tables.Unnamed indicates further investigation is required.

returns_raw.columns.tolist()

['Order ID',
 'Returned',
 'Unnamed: 2',
 'Unnamed: 3',
 'Unnamed: 4',
 'Unnamed: 5',
 'Unnamed: 6']

In [10]:
# Inspect the Orders table and data types.
people.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Regional Manager  4 non-null      str  
 1   Region            4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes


In [11]:
# Additional Unnamed columns in returns schema is due to outlier value
# .copy() creates a separate clean DataFrame without changing returns_raw.

returns_clean = returns_raw.loc[:, ["Order ID", "Returned"]].copy()

In [12]:
# Validating the dimensions after removing the unwanted columns.

returns_clean.shape

(296, 2)

In [13]:
# Checking missing values in the two retained columns.
returns_clean.isna().sum()

Order ID    0
Returned    0
dtype: int64

In [14]:
# Rechecking duplication of order ID in reurns_clean.

returns_clean["Order ID"].duplicated().sum()

np.int64(0)

In [15]:
# Confirm the return-status values.z
returns_clean["Returned"].value_counts(
    dropna=False
)

Returned
Yes    296
Name: count, dtype: int64

In [16]:
# Convert the columns explicitly to datetime.
# errors="raise" stops execution if pandas finds an invalid date.

orders["Order Date"] = pd.to_datetime(  orders["Order Date"],errors="raise")

orders["Ship Date"] = pd.to_datetime(orders["Ship Date"],errors="raise")

In [17]:
orders[["Order Date", "Ship Date"]].dtypes

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [18]:
# Counting the records where the product was shipped before it was ordered.
# A result of zero indicates a valid date sequence.

invalid_shipping_dates = (orders["Ship Date"] < orders["Order Date"]).sum()

invalid_shipping_dates

np.int64(0)

In [19]:
# Summarise the important key identifiers.

key_validation = pd.Series({
    "order_line_rows": len(orders),
    "unique_row_ids": orders["Row ID"].nunique(),
    "unique_orders": orders["Order ID"].nunique(),
    "returns_order_id_unique": returns_clean["Order ID"].is_unique,
    "returned_orders": returns_clean["Order ID"].nunique(),
    "people_region_unique": people["Region"].is_unique,
    "regions_in_people": people["Region"].nunique()
})

key_validation

order_line_rows            10194
unique_row_ids             10194
unique_orders               5111
returns_order_id_unique     True
returned_orders              296
people_region_unique        True
regions_in_people              4
dtype: object

In [20]:
# Orders contains multiple product rows for an Order ID,
# while Returns contains one row per returned Order ID.
# Therefore, the relationship is many-to-one.

orders_enriched = orders.merge(
    returns_clean,
    on="Order ID",
    how="left",
    validate="many_to_one"
)

# As a result,unmatched orders were not returned.

orders_enriched["Returned"] = (
    orders_enriched["Returned"].fillna("No")
)

In [21]:
# Each region maps to one regional manager.

orders_enriched = orders_enriched.merge(
    people,
    on="Region",
    how="left",
    validate="many_to_one"
)

In [22]:
orders_enriched.shape

(10194, 23)

In [23]:
orders_enriched["Returned"].value_counts(dropna=False)

Returned
No     9394
Yes     800
Name: count, dtype: int64

In [24]:
# Every region have a manager.
orders_enriched["Regional Manager"].isna().sum()

np.int64(0)

In [25]:
# Reconfirmed that merging did not change the row count,total sales or total profit.

reconciliation = pd.Series({
    "rows_before_merge": len(orders),
    "rows_after_merge": len(orders_enriched),
    "sales_before_merge": orders["Sales"].sum(),
    "sales_after_merge": orders_enriched["Sales"].sum(),
    "profit_before_merge": orders["Profit"].sum(),
    "profit_after_merge": orders_enriched["Profit"].sum()
})

reconciliation

rows_before_merge      1.019400e+04
rows_after_merge       1.019400e+04
sales_before_merge     2.326534e+06
sales_after_merge      2.326534e+06
profit_before_merge    2.922968e+05
profit_after_merge     2.922968e+05
dtype: float64

In [26]:
# These assertions ensure that the merges did not duplicate or remove sales records.

assert len(orders_enriched) == len(orders)

assert np.isclose(
    orders_enriched["Sales"].sum(),
    orders["Sales"].sum()
)

assert np.isclose(
    orders_enriched["Profit"].sum(),
    orders["Profit"].sum()
)

assert orders_enriched["Returned"].isna().sum() == 0

assert (
    orders_enriched["Regional Manager"]
    .isna()
    .sum()
    == 0
)

In [27]:
# Save the reusable cleaned dataset

PROCESSED_FILE = (
    PROCESSED_DATA_DIR / "orders_enriched.csv"
)

orders_enriched.to_csv(
    PROCESSED_FILE,
    index=False
)

PROCESSED_FILE.exists()

True